<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Regional Analysis for Lead Generation</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Find the <strong>EP&nbsp;patent applicants in your region</strong> &mdash; and see which of them are worth approaching for IP services.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 620px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What you will do in this notebook</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Check the data edition
            <br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Find your region's NUTS codes (both vintages)
            <br/>Step&nbsp;3 &nbsp;&middot;&nbsp; Set your region &amp; time window
            <br/>Step&nbsp;4 &nbsp;&middot;&nbsp; <strong>Axis 1 &mdash; portfolio depth</strong> (the ranked company list)
            <br/>Step&nbsp;5 &nbsp;&middot;&nbsp; <strong>Axis 2 &mdash; geographic reach</strong>
            <br/>Step&nbsp;6 &nbsp;&middot;&nbsp; <strong>Segment</strong> the companies into lead tiers
            <br/>Step&nbsp;7 &nbsp;&middot;&nbsp; Know what this can &amp; cannot see
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 620px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Run the cells top to bottom.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            The default example is <strong>Alsace (FR42)</strong> and runs out of the box.
            To analyse your own region, change the <code>NUTS_CODES</code> in Step&nbsp;3.
            Every query runs directly on PATSTAT inside EPO&nbsp;TIP &mdash; no BigQuery, no extra credentials.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global, Autumn 2025.
    </div>
</div>

---

## Why this notebook

Your PATLIB is asked to justify its outreach budget. Which companies in your region should you
actually be talking to, and why those? Every PATLIB has a contact list, and almost none can say
how it was built — who walked in, who came to the last event, who somebody remembered. The
question a ranked list has to survive is not *"who is on it"* but *"who is missing"*.

This notebook builds the list from the patent record instead: every EP and PCT applicant in your
region, placed on two axes — how deep the portfolio is, how far it reaches — and segmented into
tiers you can act on. Alsace is the default so it runs out of the box; change one line for your
own region.

**At the end you have** a named shortlist for your region, the grid it came from, and one
defensible sentence about what the list does not contain.


## Setup: Connect to PATSTAT

Run this cell first. It connects to PATSTAT on EPO TIP and defines a small `run_query`
helper that runs SQL and returns the result as a pandas DataFrame with timing info.

In [ ]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# Connect to PATSTAT (PROD = the full production database on TIP)
patstat = PatstatClient(env='PROD')

def run_query(query):
    """Execute SQL on PATSTAT and return a DataFrame with timing info."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    elapsed = time.time() - start
    df = pd.DataFrame(res)
    print(f"Query took {elapsed:.2f}s ({len(df)} rows)")
    return df

print("Connected to PATSTAT. Ready to run queries.")

---

## Step 1: Which data edition are we on?

**Why:** Every number in this notebook depends on the PATSTAT edition. Before analysing,
confirm how fresh the data is by asking for the most recent filing date. On the current
**Autumn 2025** edition you should see roughly **2025-09-23**.

In [ ]:
df_edition = run_query("""
SELECT MAX(appln_filing_date) AS latest_filing_date
FROM tls201_appln
WHERE appln_filing_year BETWEEN 2024 AND 2026
""")
df_edition

---

## Step 2: Find your region's NUTS codes

**Why:** We locate applicants by their **NUTS** region code. But PATSTAT stores **two NUTS
vintages side by side**, and a region has a *different code in each*:

- **level-3** records carry the **older** codes and come *with a name label* &mdash; e.g. Alsace = `FR421` (Bas-Rhin) + `FR422` (Haut-Rhin).
- **level-4** records carry the **current** REGPAT codes and have *no label* in PATSTAT &mdash; e.g. Alsace = `FRF11` + `FRF12`.

If you filter on only one of them (for example `SUBSTR(nuts,1,4) = 'FR42'`), you silently
drop every record stored under the other vintage. For Alsace that is the difference between
**52 companies / 280 families** and the correct **78 / 396**.

The query below lists the NUTS codes actually present in the data for a country, at both
levels, with the label where one exists. Use it to read off **all** codes for your region
&mdash; the labelled level-3 ones *and* their unlabelled level-4 counterparts &mdash; then put
them into Step 3.

**Try it:** change `COUNTRY` to your country (`FR`, `IT`, `BE`, `DE`, `PL`) and narrow the
`nuts LIKE` prefix to the area you care about.

In [ ]:
# --- CHANGE THIS -------------------------------------------------
COUNTRY  = 'FR'              # FR, IT, BE, DE, PL ...
PREFIXES = ['FR42', 'FRF1']  # broad prefixes that bracket your region (both vintages)
# -----------------------------------------------------------------

prefix_filter = " OR ".join(f"p.nuts LIKE '{pre}%'" for pre in PREFIXES)

df_nuts = run_query(f"""
SELECT
    p.nuts,
    p.nuts_level,
    n.nuts_label,
    COUNT(DISTINCT p.person_id) AS person_records
FROM tls206_person p
LEFT JOIN tls904_nuts n ON n.nuts = p.nuts        -- LEFT JOIN: level-4 codes have no label
WHERE p.person_ctry_code = '{COUNTRY}'
  AND p.nuts_level IN (3, 4)
  AND ({prefix_filter})
GROUP BY p.nuts, p.nuts_level, n.nuts_label
ORDER BY p.nuts
""")
df_nuts

### What to look for

You will typically see **labelled level-3 codes** (the region names you recognise) and
**unlabelled level-4 codes** (`nuts_label` is `None`) for the *same* geography. For Alsace:

| code | level | label | vintage |
|---|---|---|---|
| `FR421` | 3 | Bas-Rhin | old |
| `FR422` | 3 | Haut-Rhin | old |
| `FRF11` | 4 | *(none)* | current |
| `FRF12` | 4 | *(none)* | current |

Collect **all four** &mdash; both vintages &mdash; for the next step. (Level-4 codes are the
official NUTS-2021 codes; if you are unsure which level-4 code maps to your region, look it
up in the Eurostat NUTS-2021 classification.)

---

## Step 3: Set your region and time window

**This is the only cell you need to edit.** Put **all** the NUTS codes for your region here
(from Step 2) and choose a filing-year window. The default is **Alsace, 2017&ndash;2022**,
which reproduces the verified reference figures.

> **How many codes do you need?** It depends on the country. **France** renumbered its
> regions between the NUTS 2016 and 2021 vintages, so Alsace needs *four* codes
> (`FR421`,`FR422` **and** `FRF11`,`FRF12`). **Germany** kept its codes stable, so a whole
> Bundesland is a *single* prefix &mdash; e.g. Saxony = `DED`. Step&nbsp;2 always shows you
> the truth for your region: list every code it returns.

> **A note on speed.** The default **Alsace** example runs in a few seconds. A whole large
> Bundesland (e.g. Bavaria `DE2`, North&nbsp;Rhine-Westphalia `DEA`) covers far more
> applicants and families, so Steps&nbsp;5&ndash;6 scan more data and can take noticeably
> longer &mdash; that is expected, not an error. Queries on TIP carry no query cost.

In [ ]:
# --- CHANGE THIS: your region and window --------------------------
NUTS_CODES = ['FR421', 'FR422', 'FRF11', 'FRF12']   # Alsace = Bas-Rhin + Haut-Rhin (both vintages)
YEAR_START = 2017
YEAR_END   = 2022
# ------------------------------------------------------------------

# Build the region filter: match any listed NUTS code as a prefix.
NUTS_WHERE = "(" + " OR ".join(f"p.nuts LIKE '{c}%'" for c in NUTS_CODES) + ")"

print("Region filter :", NUTS_WHERE)
print("Filing window :", YEAR_START, "-", YEAR_END)

---

## Step 4: Axis 1 &mdash; portfolio depth (the company list)

**Why:** This is the core deliverable &mdash; the **ranked list of company applicants based in
your region**, each with its number of **patent families** (portfolio depth). We count
*families*, not applications, so a single invention filed in ten countries counts once.

The corpus rules (the "non-negotiables") are all in the `WHERE` clause:

- `pa.applt_seq_nr > 0` &rarr; **applicants** only (not inventors)
- `p.psn_sector = 'COMPANY'` &rarr; **companies** only (not universities/individuals)
- `p.nuts_level IN (3,4)` + `NUTS_WHERE` &rarr; **based in the region**, both vintages
- `COUNT(DISTINCT a.docdb_family_id)` &rarr; **families**, not filings

In [ ]:
df_depth = run_query(f"""
SELECT
    p.han_name                        AS applicant,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls206_person p
JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
WHERE pa.applt_seq_nr > 0
  AND p.nuts_level IN (3, 4)
  AND {NUTS_WHERE}
  AND p.psn_sector = 'COMPANY'
  AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
GROUP BY applicant
ORDER BY families DESC
""")

print(f"{len(df_depth)} companies based in the region")
df_depth.head(15)

### How to read this

For **Alsace 2017&ndash;2022** you should get **78 companies / 396 families**, led by
**HAGER ELECTRO SAS (63)** and **KUHN SAS (38)**. This is the regional "SME pyramid": a few
large filers at the top and a long tail of one- and two-family companies. Run the next cell
to see that shape.

In [ ]:
df_dist = run_query(f"""
WITH depth AS (
  SELECT p.han_name AS applicant, COUNT(DISTINCT a.docdb_family_id) AS families
  FROM tls206_person p
  JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
  JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
  WHERE pa.applt_seq_nr > 0
    AND p.nuts_level IN (3, 4)
    AND {NUTS_WHERE}
    AND p.psn_sector = 'COMPANY'
    AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
  GROUP BY applicant
)
SELECT
  CASE
    WHEN families = 1              THEN '1'
    WHEN families = 2              THEN '2'
    WHEN families BETWEEN 3  AND 4  THEN '3-4'
    WHEN families BETWEEN 5  AND 10 THEN '5-10'
    WHEN families BETWEEN 11 AND 20 THEN '11-20'
    WHEN families BETWEEN 21 AND 50 THEN '21-50'
    ELSE '>50'
  END               AS family_class,
  COUNT(*)          AS companies
FROM depth
GROUP BY family_class
ORDER BY MIN(families)
""")
df_dist

---

## Step 5: Axis 2 &mdash; geographic reach

**Why:** Depth tells you *how much* a company files; reach tells you *how far* it protects
its inventions. For each company we take **all members of each of its families** and see
which economic zones they reach:

- **North America** &mdash; US, CA
- **Asia** &mdash; CN, JP, KR, IN, TW, SG, IL
- **Oceania** &mdash; AU, NZ

(EP and WO themselves are the European/PCT *route* almost every company here uses, so the
interesting signal is whether a portfolio also reaches *beyond* Europe.)

Note the second join back to `tls201_appln` on `docdb_family_id`: that is what lets us read
the authorities of **every family member**, not just the regional filing.

In [ ]:
df_reach = run_query(f"""
WITH corp_fam AS (          -- one row per (company, family) in the region
  SELECT DISTINCT p.han_name AS applicant, a.docdb_family_id
  FROM tls206_person p
  JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
  JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
  WHERE pa.applt_seq_nr > 0
    AND p.nuts_level IN (3, 4)
    AND {NUTS_WHERE}
    AND p.psn_sector = 'COMPANY'
    AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
),
fam_zone AS (               -- for each family: does ANY member reach each zone?
  SELECT cf.applicant, cf.docdb_family_id,
    MAX(CASE WHEN m.appln_auth IN ('US','CA')                          THEN 1 ELSE 0 END) AS z_na,
    MAX(CASE WHEN m.appln_auth IN ('CN','JP','KR','IN','TW','SG','IL') THEN 1 ELSE 0 END) AS z_asia,
    MAX(CASE WHEN m.appln_auth IN ('AU','NZ')                          THEN 1 ELSE 0 END) AS z_oce
  FROM corp_fam cf
  JOIN tls201_appln m ON m.docdb_family_id = cf.docdb_family_id
  GROUP BY cf.applicant, cf.docdb_family_id
)
SELECT
  applicant,
  COUNT(*)      AS families,
  SUM(z_na)     AS fam_north_america,
  SUM(z_asia)   AS fam_asia,
  SUM(z_oce)    AS fam_oceania
FROM fam_zone
GROUP BY applicant
ORDER BY families DESC
""")
df_reach.head(15)

### How to read this

Each column counts *how many of a company's families* reach that zone. For Alsace,
**HAGER ELECTRO** protects broadly in Asia (35 families) and Oceania (20); **KUHN** and
**CRYOSTAR** lean North-American. A company whose families almost never leave Europe is a
different kind of lead from one with worldwide coverage &mdash; which is exactly what we
turn into tiers next.

---

## Step 6: Segment into lead tiers (depth &times; reach)

**Why:** Now combine the two axes into a simple grid a PATLIB can act on. We use **neutral
tiers**:

- **Depth** &mdash; `small` (1&ndash;2 families), `medium` (3&ndash;10), `large` (&gt;10)
- **Reach** &mdash; `local` (each family uses a single filing route only), `regional`
  (a family spans two routes/zones but stays within Europe/PCT), `global` (at least one
  family reaches North America, Asia or Oceania)

Read the grid as lead priority: **large&times;global** firms are candidates to *invite as
speakers/partners*; **medium/small&times;global or regional** firms are growing filers who
often *need IP services and training*; **small&times;local** firms are early-stage contacts.

In [ ]:
df_segments = run_query(f"""
WITH corp_fam AS (
  SELECT DISTINCT p.han_name AS applicant, a.docdb_family_id
  FROM tls206_person p
  JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
  JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
  WHERE pa.applt_seq_nr > 0
    AND p.nuts_level IN (3, 4)
    AND {NUTS_WHERE}
    AND p.psn_sector = 'COMPANY'
    AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
),
fam_zone AS (
  SELECT cf.applicant, cf.docdb_family_id,
    MAX(CASE WHEN m.appln_auth = 'EP'                                  THEN 1 ELSE 0 END) AS z_ep,
    MAX(CASE WHEN m.appln_auth = 'WO'                                  THEN 1 ELSE 0 END) AS z_wo,
    MAX(CASE WHEN m.appln_auth IN ('US','CA')                          THEN 1 ELSE 0 END) AS z_na,
    MAX(CASE WHEN m.appln_auth IN ('CN','JP','KR','IN','TW','SG','IL') THEN 1 ELSE 0 END) AS z_asia,
    MAX(CASE WHEN m.appln_auth IN ('AU','NZ')                          THEN 1 ELSE 0 END) AS z_oce
  FROM corp_fam cf
  JOIN tls201_appln m ON m.docdb_family_id = cf.docdb_family_id
  GROUP BY cf.applicant, cf.docdb_family_id
),
company AS (
  SELECT applicant,
    COUNT(*)                                          AS families,
    MAX(z_ep + z_wo + z_na + z_asia + z_oce)          AS widest_family_zones,
    SUM(z_na) + SUM(z_asia) + SUM(z_oce)              AS beyond_europe_hits
  FROM fam_zone
  GROUP BY applicant
)
SELECT
  CASE WHEN families > 10 THEN 'large'
       WHEN families BETWEEN 3 AND 10 THEN 'medium'
       ELSE 'small' END                                   AS depth_tier,
  CASE WHEN beyond_europe_hits > 0 THEN 'global'
       WHEN widest_family_zones >= 2 THEN 'regional'
       ELSE 'local' END                                   AS reach_tier,
  COUNT(*)                                                AS companies
FROM company
GROUP BY depth_tier, reach_tier
ORDER BY depth_tier, reach_tier
""")

# pivot into a readable depth x reach grid (pivot_table handles missing tier combos)
grid = (pd.pivot_table(df_segments, index='depth_tier', columns='reach_tier',
                       values='companies', aggfunc='sum', fill_value=0)
        .reindex(index=['small', 'medium', 'large'],
                 columns=['local', 'regional', 'global'], fill_value=0)
        .astype(int))
print("companies by depth (rows) x reach (cols) -- total:", int(df_segments['companies'].sum()))
grid

### How to read this

The grid totals back to your company count (**78** for Alsace) — a quick map of how the
region's companies split across the tiers. Read it as lead priority; the next cell turns
that map into the actual **named shortlist** you can act on.

### Your shortlist — every company with its tier

**Why:** the grid *counts*; this is the list you *act on*. One row per company, with its
family count, its **depth** and **reach** tier, and how many families reach each zone —
biggest first. Filter it to the tier you want to approach (see the commented example).

In [ ]:
df_leads = run_query(f"""
WITH corp_fam AS (
  SELECT DISTINCT p.han_name AS applicant, a.docdb_family_id
  FROM tls206_person p
  JOIN tls207_pers_appln pa ON p.person_id = pa.person_id
  JOIN tls201_appln a       ON pa.appln_id  = a.appln_id
  WHERE pa.applt_seq_nr > 0
    AND p.nuts_level IN (3, 4)
    AND {NUTS_WHERE}
    AND p.psn_sector = 'COMPANY'
    AND a.appln_filing_year BETWEEN {YEAR_START} AND {YEAR_END}
),
fam_zone AS (
  SELECT cf.applicant, cf.docdb_family_id,
    MAX(CASE WHEN m.appln_auth = 'EP'                                  THEN 1 ELSE 0 END) AS z_ep,
    MAX(CASE WHEN m.appln_auth = 'WO'                                  THEN 1 ELSE 0 END) AS z_wo,
    MAX(CASE WHEN m.appln_auth IN ('US','CA')                          THEN 1 ELSE 0 END) AS z_na,
    MAX(CASE WHEN m.appln_auth IN ('CN','JP','KR','IN','TW','SG','IL') THEN 1 ELSE 0 END) AS z_asia,
    MAX(CASE WHEN m.appln_auth IN ('AU','NZ')                          THEN 1 ELSE 0 END) AS z_oce
  FROM corp_fam cf
  JOIN tls201_appln m ON m.docdb_family_id = cf.docdb_family_id
  GROUP BY cf.applicant, cf.docdb_family_id
),
company AS (
  SELECT applicant,
    COUNT(*)                                 AS families,
    MAX(z_ep + z_wo + z_na + z_asia + z_oce) AS widest_family_zones,
    SUM(z_na)   AS fam_north_america,
    SUM(z_asia) AS fam_asia,
    SUM(z_oce)  AS fam_oceania,
    SUM(z_na) + SUM(z_asia) + SUM(z_oce)     AS beyond_europe_hits
  FROM fam_zone
  GROUP BY applicant
)
SELECT
  applicant,
  families,
  CASE WHEN families > 10 THEN 'large'
       WHEN families BETWEEN 3 AND 10 THEN 'medium'
       ELSE 'small' END                       AS depth_tier,
  CASE WHEN beyond_europe_hits > 0 THEN 'global'
       WHEN widest_family_zones >= 2 THEN 'regional'
       ELSE 'local' END                        AS reach_tier,
  fam_north_america,
  fam_asia,
  fam_oceania
FROM company
ORDER BY families DESC
""")

# Example — the large-portfolio, globally-active leads (candidates to invite as speakers/partners):
# df_leads[(df_leads.depth_tier == 'large') & (df_leads.reach_tier == 'global')]

df_leads.head(20)

---

## Step 7: What this can &mdash; and cannot &mdash; see

<div style="background:#fffbeb; border:1px solid #fcd34d; border-radius:10px; padding:16px 20px;">
<strong>&#9888; A NUTS filter finds only the EP/PCT-active companies of a region.</strong>
<br/><br/>
NUTS region codes are attached <em>only</em> on the European/PCT route (EPO at level&nbsp;3,
OECD REGPAT at level&nbsp;4). Purely national filings &mdash; e.g. a French application that
never goes to the EPO or via PCT &mdash; carry <em>no</em> NUTS code. Measured on the current
edition, roughly <strong>70% of national patent families never take the EP/PCT route</strong>,
and about <strong>77% of company applicant records have no NUTS at all</strong>. Those are
typically the smaller, locally-filing firms.
<br/><br/>
So this list is the <strong>EP/PCT-active subset</strong> of your region &mdash; excellent for
finding internationally-minded filers, but it does <em>not</em> see the national-only tail.
PATSTAT cannot recover them by postcode either (the structured ZIP field is empty and
addresses are sparse). For the <em>full</em> regional population you would add national-office
data (INPI, DPMA/DEPATISnet) or an external city&rarr;region lookup.
</div>

**One more caveat for a final ranking:** `han_name` sometimes splits one group into several
rows (e.g. `HAGER ELECTRO` vs `HAGER CONTROLS`; `KUHN SAS` vs `KUHN SA`). Before you publish a
league table, consolidate via `doc_std_name_id` / `psn_id` &mdash; see the harmonisation guide.

---

## Try it yourself

1. **Another region:** go to **Step&nbsp;2**, set `COUNTRY` and the `PREFIXES` for your area,
   read off *all* the NUTS codes it returns, and paste them into `NUTS_CODES` in **Step&nbsp;3**.
2. **Another window:** change `YEAR_START` / `YEAR_END` in Step&nbsp;3 (a 5&ndash;6 year window
   works well &mdash; long enough for a portfolio to show, recent enough to be current).
3. Re-run Steps 4&ndash;6.

### France &mdash; needs both NUTS vintages
Alsace is the default. A French region carries *old* level-3 codes **and** *new* level-4
codes, so list all of them:

```python
NUTS_CODES = ['FR421', 'FR422', 'FRF11', 'FRF12']   # Alsace
```

### Germany &mdash; one prefix per Bundesland
German codes are stable across vintages, so a whole **Bundesland** is a single prefix. Pick
one and set e.g. `NUTS_CODES = ['DED']` (Saxony):

> *Known-good check:* Saxony (`DED`, 2017&ndash;2022) returns **287 companies / 920 families**, led by NOVALED GMBH (155). If your run matches, your region swap worked.

| Code | Bundesland | | Code | Bundesland |
|---|---|---|---|---|
| `DE1` | Baden-W&uuml;rttemberg | | `DE9` | Niedersachsen |
| `DE2` | Bayern | | `DEA` | Nordrhein-Westfalen |
| `DE3` | Berlin | | `DEB` | Rheinland-Pfalz |
| `DE4` | Brandenburg | | `DEC` | Saarland |
| `DE5` | Bremen | | `DED` | Sachsen |
| `DE6` | Hamburg | | `DEE` | Sachsen-Anhalt |
| `DE7` | Hessen | | `DEF` | Schleswig-Holstein |
| `DE8` | Mecklenburg-Vorpommern | | `DEG` | Th&uuml;ringen |

For a single *Kreis* instead of a whole Bundesland, use a longer prefix from Step&nbsp;2
(e.g. `DED2` = Dresden area). Other countries (`IT`, `BE`, `PL`) work the same way &mdash;
let Step&nbsp;2 tell you which codes exist.

*Data: EPO PATSTAT Global, Autumn 2025 &middot; mtc.berlin*